# ParkWise — retrain YOLOv5s on PKLot

Runtime → **Change runtime type → GPU** (A100 or L4 on Colab Pro). Run cells top to bottom.

**Before you start:**
1. Colab **Secrets** (🔑 icon in the left sidebar) → add `ROBOFLOW_API_KEY` from roboflow.com → Settings → API. Never paste the key into a cell.
2. Upload `models/best-2022.pt` from the repo to your Drive at `MyDrive/parkwise-runs/best-2022.pt` (for the before/after comparison in step 6).
3. Push the repo's `main` — step 4 fetches `remap_labels.py` from GitHub.

Everything lands in `MyDrive/parkwise-runs/`, so a disconnect loses nothing; re-run the train cell with `--resume` if it happens.

In [ ]:
# 1. Drive + working dir
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/parkwise-runs'
!mkdir -p {RUNS}
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Toolchain — pinned to match tools/export/.venv so the checkpoint round-trips.
#    torch >= 2.6 flips torch.load to weights_only=True, which breaks yolov5 v7.0
#    loading the pretrained yolov5s.pt. Pinning avoids patching anything.
%cd /content
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!git clone -q --branch v7.0 --depth 1 https://github.com/ultralytics/yolov5
!pip install -q -r yolov5/requirements.txt "numpy<2" roboflow
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 3. Download PKLot in YOLOv5 format.
#    ⚠ CONFIRM THESE THREE VALUES from the Universe page: open the dataset →
#    Download → "YOLO v5 PyTorch" → "show download code". Use the 640 version
#    (pre-resized) rather than raw.
WORKSPACE = 'brad-dwyer'
PROJECT   = 'pklot-1tros'
VERSION   = 2

from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov5', location='/content/pklot')
DATA = dataset.location
print('dataset at', DATA)
!cat {DATA}/data.yaml
!echo && echo train: $(ls {DATA}/train/images | wc -l)  valid: $(ls {DATA}/valid/images | wc -l)  test: $(ls {DATA}/test/images 2>/dev/null | wc -l)

In [ ]:
# 4. Remap classes so index 0 = Occupied, matching the app's invariant.
#    Roboflow orders alphabetically (space-empty first) — the opposite of what
#    the app expects. Flipping the DATASET keeps every assert in the repo intact.
!wget -q -O remap_labels.py https://raw.githubusercontent.com/arya-pradhan/ParkWise/main/tools/train/remap_labels.py
!python remap_labels.py {DATA}
!echo && cat {DATA}/data.yaml

In [ ]:
# 5. Train. ~1 min/epoch on an A100; --patience stops early once mAP plateaus.
#    If the session drops: re-run with  --resume {RUNS}/pklot/weights/last.pt
%cd /content/yolov5
!python train.py --img 640 --batch 32 --epochs 50 \
  --data {DATA}/data.yaml --weights yolov5s.pt \
  --project {RUNS} --name pklot --exist-ok \
  --cache ram --patience 15

In [ ]:
# 6. Before/after on the SAME test split — the honest number for the model card.
#    (2022 model at its native 416; new model at 640.)
%cd /content/yolov5
print('=== 2022 model ===')
!python val.py --weights {RUNS}/best-2022.pt --data {DATA}/data.yaml --img 416 --task test --name val-2022 --project {RUNS} --exist-ok
print('\n=== 2026 model ===')
!python val.py --weights {RUNS}/pklot/weights/best.pt --data {DATA}/data.yaml --img 640 --task test --name val-2026 --project {RUNS} --exist-ok
print('\nCopy the "all" rows (P, R, mAP50, mAP50-95) from both into the model card.')

In [ ]:
# 7. Pick ~8 test images for the site's sample gallery: spread across lots and
#    weather. PKLot filenames encode lot + weather + date; sample by prefix.
import os, random, shutil, re, collections
random.seed(7)
src = f'{DATA}/test/images'
files = sorted(os.listdir(src))
groups = collections.defaultdict(list)
for f in files:
    m = re.match(r'(\d{4}-\d{2}-\d{2}_\d{2}_\d{2}_\d{2}|[A-Za-z0-9]+)', f)
    key = f.split('_')[0][:10]
    groups[key].append(f)
keys = sorted(groups)
print(f'{len(files)} test images in {len(keys)} groups; showing a few keys:', keys[:6])
picked = []
for k in random.sample(keys, min(8, len(keys))):
    picked.append(random.choice(groups[k]))
out = f'{RUNS}/samples'
os.makedirs(out, exist_ok=True)
for i, f in enumerate(picked):
    shutil.copy(f'{src}/{f}', f'{out}/pklot-{i+1:02d}.jpg')
print('copied', len(picked), 'samples ->', out)

In [ ]:
# 8. Bundle what the repo needs. Download parkwise-retrain.zip from Drive.
!cd {RUNS} && zip -q -r parkwise-retrain.zip \
  pklot/weights/best.pt pklot/results.csv pklot/confusion_matrix.png pklot/val_batch0_pred.jpg pklot/val_batch1_pred.jpg \
  samples val-2022 val-2026
!ls -la {RUNS}/parkwise-retrain.zip
print('\nNext, locally:')
print('  unzip -> models/best.pt, public/model-card/, public/samples/pklot-*.jpg')
print('  then follow tools/train/README.md → "After training"')